In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
import pickle

In [122]:
with open("../data/final_drugs_df","rb") as f:
    final_drugs_df =pickle.load(f)
with open("../data/final_indications_df","rb") as f:
    final_indications_df = pickle.load(f)
with open("../data/final_diseases_df","rb") as f:
    final_diseases_df =pickle.load(f)
with open("../data/final_trials_df","rb") as f:
    final_trials_df =pickle.load(f)
with open("../data/embeddings_df","rb") as f:
    embeddings_df =pickle.load(f)

In [123]:
final_diseases_df["disease_id"] = final_diseases_df["disease_id"].str.replace("_",":")
embeddings_df["disease_id"] = embeddings_df["disease_id"].str.replace("_",":")
final_indications_df["disease_id"] = final_indications_df["efo_id"]
merged_df = final_indications_df.merge(final_drugs_df, on="drug_id").merge(final_diseases_df, on="disease_id").merge(embeddings_df, on =["disease_id", "drug_id"])

In [132]:
# TODO: generate other features
for i, row in merged_df.iterrows():
    merged_df.at[i, "n_shared_pathways"] = len(set(merged_df.loc[i,"drug_pathways"]) and set(merged_df.loc[i,"disease_pathways"]))

In [134]:
for i, row in merged_df.iterrows():
    if merged_df.loc[i,"max_phase_for_ind"] == "4.0":
        merged_df.at[i, "overall_success"] = True
    elif merged_df.loc[i,"nct_evidence"]:
        all_results = merged_df.loc[i,"nct_evidence"]
        highest_phase = np.max([int(result[-1]) for result in all_results])
        latest_results = [result[:-1] for result in all_results if int(result[-1]) == highest_phase]
        
        if "success" in latest_results and not "fail" in latest_results:
            merged_df.at[i, "overall_success"] = True
        elif "fail" in latest_results:
            merged_df.at[i, "overall_success"] = False
        else:
            merged_df.at[i, "overall_success"] = None
    else:
        merged_df.at[i, "overall_success"] = None
merged_df["overall_success"] = merged_df["overall_success"].astype("boolean")

first_columns = ["drug_id", "disease_id", "overall_success", "nct_evidence", "max_phase_for_ind"] 
merged_df= merged_df[first_columns + [c for c in merged_df.columns if c not in first_columns]]

In [135]:
merged_df = merged_df.select_dtypes(exclude=["object"])
merged_df = merged_df.drop(columns=["first_approval", "withdrawn_flag", "usan_year", "black_box_warning", "first_in_class", "name_similarity"])

In [136]:
merged_df = merged_df[ ~merged_df["biotherapeutic"].astype(bool) ]
merged_df = merged_df.rename(columns={"overall_success" : "label"})
merged_df = merged_df[merged_df.label.notna() ]
merged_df

,label,biotherapeutic,chemical_probe,dosed_ingredient,inorganic_flag,natural_product,oral,orphan,parenteral,prodrug,therapeutic_flag,topical,veterinary,aromatic_rings,hba,hbd,heavy_atoms,num_ro5_violations,rtb,n_shared_pathways
4,True,0,0,True,0,0,True,0,False,0,True,False,0,3.0,4.0,1.0,26.0,0.0,3.0,58.0
5,True,0,0,True,0,0,True,0,False,0,True,False,0,3.0,4.0,1.0,26.0,0.0,3.0,71.0
13,True,0,0,True,0,1,True,0,True,0,True,False,0,3.0,10.0,5.0,33.0,0.0,9.0,71.0
14,True,0,0,True,0,1,True,0,False,0,True,False,0,2.0,2.0,1.0,17.0,0.0,3.0,71.0
15,True,0,0,True,0,1,True,0,False,0,True,True,1,0.0,5.0,3.0,26.0,0.0,2.0,71.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1129,True,0,0,True,0,1,True,0,False,0,True,False,0,1.0,2.0,1.0,23.0,1.0,4.0,71.0
1157,True,0,0,True,0,0,True,0,False,0,True,False,0,1.0,3.0,2.0,8.0,0.0,0.0,30.0
1158,True,0,0,True,0,1,True,1,False,0,True,False,0,1.0,2.0,2.0,23.0,1.0,6.0,71.0
1180,True,0,0,True,0,0,True,1,False,0,True,False,0,3.0,4.0,2.0,31.0,0.0,7.0,29.0


In [145]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# ---------------------------------------
# 1. Split predictors / label
# ---------------------------------------
X = merged_df.drop(columns=["label"])
y = merged_df["label"]

# Identify column types
num_cols = X.select_dtypes(include=["float64", "int64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# ---------------------------------------
# 2. Preprocessing pipelines
# ---------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# ---------------------------------------
# 3. Models
# ---------------------------------------
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42))
])

logreg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("logreg", LogisticRegression(max_iter=500))
])

# ---------------------------------------
# 4. Train/test split
# ---------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------
# 5. Fit models
# ---------------------------------------
rf_model.fit(X_train, y_train)
logreg_model.fit(X_train, y_train)

# ---------------------------------------
# 6. Predictions
# ---------------------------------------
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

logreg_pred = logreg_model.predict(X_test)
logreg_proba = logreg_model.predict_proba(X_test)[:, 1]

# ---------------------------------------
# 7. Evaluation
# ---------------------------------------
print("Random Forest:")
print("Confusion matrix:\n", confusion_matrix(y_test, rf_pred))
print("  Accuracy:", accuracy_score(y_test, rf_pred))
print("  AUC:", roc_auc_score(y_test, rf_proba))

print("\nLogistic Regression:")
print("Confusion matrix:\n", confusion_matrix(y_test, logreg_pred))
print("  Accuracy:", accuracy_score(y_test, logreg_pred))
print("  AUC:", roc_auc_score(y_test, logreg_proba))

Random Forest:
Confusion matrix:
 [[ 3  3]
 [ 1 24]]
  Accuracy: 0.8709677419354839
  AUC: 0.9

Logistic Regression:
Confusion matrix:
 [[ 3  3]
 [ 0 25]]
  Accuracy: 0.9032258064516129
  AUC: 0.7933333333333334


In [ ]:
import numpy as np

# Extract trained RF model inside the pipeline
rf = rf_model.named_steps["rf"]

# Get the transformed column names
num_features = list(num_cols)
cat_features = list(cat_cols)

# After transformation, numeric features stay single.
# Categorical features remain single because we didn't one-hot encode them.
all_features = num_features + cat_features

# Extract importances
importances = rf.feature_importances_

# Combine into one dataframe
rf_feature_importance = pd.DataFrame({
    "feature": all_features,
    "importance": importances
}).sort_values(by="importance", ascending=False)
print(rf_feature_importance) 

               feature  importance
13   n_shared_pathways    0.297676
12                 rtb    0.139689
10         heavy_atoms    0.120644
9                  hbd    0.115289
7       aromatic_rings    0.099981
8                  hba    0.086556
3      natural_product    0.051866
11  num_ro5_violations    0.028921
2       inorganic_flag    0.024043
5              prodrug    0.011144
1       chemical_probe    0.009905
4               orphan    0.009450
6           veterinary    0.004836
0       biotherapeutic    0.000000


In [140]:
logreg = logreg_model.named_steps["logreg"]

coef = logreg.coef_[0]   # Binary classification → one vector

logreg_feature_importance = pd.DataFrame({
    "feature": all_features,
    "coef": coef,
    "abs_coef": np.abs(coef)
}).sort_values(by="abs_coef", ascending=False)

print(logreg_feature_importance)

               feature      coef  abs_coef
9                  hbd  1.292699  1.292699
13   n_shared_pathways  1.108114  1.108114
8                  hba -0.807709  0.807709
12                 rtb -0.803067  0.803067
11  num_ro5_violations  0.639944  0.639944
4               orphan  0.516731  0.516731
1       chemical_probe  0.507265  0.507265
3      natural_product  0.353770  0.353770
7       aromatic_rings  0.323418  0.323418
2       inorganic_flag  0.315407  0.315407
6           veterinary  0.258294  0.258294
10         heavy_atoms -0.241318  0.241318
5              prodrug  0.198047  0.198047
0       biotherapeutic  0.000000  0.000000


In [72]:
drug_targets = np.unique([
    uniprot 
    for targets in final_drugs_df["targets"] 
    if targets
    for uniprots in targets.values() 
    for uniprot in uniprots ])

disease_targets = np.unique([
    uniprot
    for uniprots in final_diseases_df["disease_targets"]
    for uniprot in uniprots
])

all_targets = np.unique(np.concat([drug_targets, disease_targets]))

In [ ]:
drug_targets.size
disease_targets.size
all_targets.size

493

In [ ]:
# TODO: explore GNN, node2vec ....
from torch_geometric.data import HeteroData

data = HeteroData()

data['drug'].x = drug_features_tensor
data['disease'].x = disease_features_tensor
data['pathway'].x = pathway_dummy_features # can be zeros

# Edges
data['drug', 'interacts_with', 'pathway'].edge_index = drug_to_pathway_edges
data['disease', 'associated_with', 'pathway'].edge_index = disease_to_pathway_edges

# Now define a heterogeneous GNN
model = HeteroGNN(...)

# Training objective: predict edges between drug and disease
# You give positive pairs (known therapeutic links) and negative pairs (random)

In [146]:
final_diseases_df

,disease_id,name,description,associatedTargets,phenotypes,disease_targets,disease_pathways
0,EFO:0000685,rheumatoid arthritis,"A chronic, systemic autoimmune disorder charac...","{'rows': [{'target': {'id': 'ENSG00000160712',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0001370'...,"[P08887, P29597, P01375, O60674, Q9Y2R2, Q9UM0...","[R-HSA-1059683, R-HSA-110056, R-HSA-112411, R-..."
1,EFO:0004991,Myasthenia gravis,"Myasthenia gravis (MG) is a rare, clinically h...","{'rows': [{'target': {'id': 'ENSG00000171385',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0000651'...,"[Q9UK17, Q9BQ31, Q8NCM2, P22303, Q8TDN2, P2245...","[R-HSA-1296072, R-HSA-5576894, R-HSA-1296072, ..."
2,EFO:0002609,juvenile idiopathic arthritis,"Juvenile idiopathic arthritis (JIA), also know...","{'rows': [{'target': {'id': 'ENSG00000179630',...",{'rows': []},"[Q8IV20, P01375, P35354, P23219, P04150, P0888...","[R-HSA-381340, R-HSA-5357786, R-HSA-5357905, R..."
3,EFO:0004826,anti-neutrophil antibody associated vasculitis,Group of systemic vasculitis with a strong ass...,"{'rows': [{'target': {'id': 'ENSG00000197249',...",{'rows': []},"[P01009, P04150, Q06203, P11836, P21730, P2083...","[R-HSA-114608, R-HSA-204005, R-HSA-381426, R-H..."
4,EFO:0005297,Granulomatosis with Polyangiitis,A small-vessel necrotizing vasculitis characte...,"{'rows': [{'target': {'id': 'ENSG00000156738',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0000509'...,"[P11836, P21730, Q06203, P05113, P04150, O1449...","[R-HSA-375276, R-HSA-418594, R-HSA-6798695, R-..."
5,EFO:0004254,membranous glomerulonephritis,A type of glomerulonephritis that is character...,"{'rows': [{'target': {'id': 'ENSG00000153246',...",{'rows': []},"[Q13018, P18564, P11836, P04150, P20839, P1226...","[R-HSA-1482788, R-HSA-1482801, R-HSA-1482839, ..."
6,EFO:0004194,IGA glomerulonephritis,Inflammation of a specific segment of glomerul...,"{'rows': [{'target': {'id': 'ENSG00000161955',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0000979'...,"[O75888, P30556, O43597, P25101, P08603, P0415...","[R-HSA-450520, R-HSA-5669034, R-HSA-375276, R-..."
7,EFO:0004719,pemphigus vulgaris,Pemphigus is a group of chronic autoimmune ski...,"{'rows': [{'target': {'id': 'ENSG00000113580',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0002960'...,"[P04150, P11836, P20839, P12268, P32926, P0137...","[R-HSA-3371497, R-HSA-383280, R-HSA-400253, R-..."
8,EFO:0000699,Sjogren syndrome,Chronic inflammatory and autoimmune disease in...,"{'rows': [{'target': {'id': 'ENSG00000128604',...",{'rows': [{'phenotypeHPO': {'id': 'HP_0000007'...,"[Q13568, P20309, P11229, Q96RJ3, Q14765, Q8TF0...","[R-HSA-877300, R-HSA-909733, R-HSA-9860276, R-..."
9,EFO:0004237,Graves disease,Graves' disease is an autoimmune disorder that...,"{'rows': [{'target': {'id': 'ENSG00000134242',...",{'rows': []},"[Q9Y2R2, P25942, P08069, P01266, P16473, Q0229...","[R-HSA-202427, R-HSA-202430, R-HSA-198933, R-H..."
